# 00 — Overview

Welcome to **eucare**, a Python library for geometric tilings, Conway operators, and origami crease patterns.

This notebook walks through the *whole* origami pipeline in a few cells:

1. **Build a tiling** — pick a geometry (Euclidean / hyperbolic / spherical) and grow a half-edge graph from prototiles.
2. **Generate a crease pattern (CP)** — apply a CP algorithm such as *shrink-rotate* (SRG) on top of the tiling.
3. **Check flat-foldability and fold** — solve an ILP for the face stacking order to obtain a valid folded state.
4. **Export** — SVG for a plotter, FOLD format for simulators, STL for 3D.

The remaining notebooks (`01`–`06`) zoom into each of these steps.

In [ ]:
import matplotlib
matplotlib.rcParams['figure.figsize'] = (5, 5)
import matplotlib.pyplot as plt
import numpy as np

import eucare as ec
from eucare import (
    conway,
    example_graphs,
    example_tilesets,
    overlap,
    plotting,
    reciprocal_figures,
    rendering,
)


def plot_g(G, ax=None, color='black', linewidth=1.0):
    """Draw the edges of a half-edge graph G on `ax` (or the current axes)."""
    if ax is None:
        ax = plt.gca()
    lines = np.array([
        [G.geometry.to_euclidean(h.orig['pos']),
         G.geometry.to_euclidean(h.dest['pos'])]
        for h in G.halfedges_representing_edges()
    ])
    plotting.plot_lines(lines, ax=ax, colors=color, linewidths=linewidth)
    plotting.set_equal_aspect(ax)
    ax.axis('off')


## Step 1: build a tiling

In [ ]:
G = example_graphs.from_tiles(example_tilesets.platonic(4), rings=3)
G.recompute_lengths_and_angles()
plot_g(G)
plt.title('Step 1: Euclidean tiling (squares, 3 rings)')
plt.show()


## Step 2: build a crease pattern via shrink-rotate

In [ ]:
from eucare.search_trees import face_bfs_tree
from eucare.reciprocal_figures import assign_this_way_by_face_z_order, make_SRG


def srg_pipeline(G):
    """Run the standard SRG pipeline: BFS z-order -> SRG -> recompute."""
    central = min(G.faces, key=lambda f: np.linalg.norm(f.midpoint()))
    central['z_order'] = 0
    for orig, dest in face_bfs_tree(central):
        dest['z_order'] = orig['z_order'] + 1
    assign_this_way_by_face_z_order(G)
    SRG = make_SRG(G)
    SRG.recompute_lengths_and_angles()
    return SRG


In [ ]:
SRG = srg_pipeline(G)
plot_g(SRG)
plt.title('Step 2: shrink-rotate crease pattern')
plt.show()


## What's a half-edge graph?

Every tiling and crease pattern in eucare is a **half-edge graph** (DCEL). Each undirected edge is split into two half-edges that link back to each other (`rev`), to the next half-edge around their face (`nex`), and to their origin and destination vertices.

This structure makes it cheap to walk faces, find neighbours, and perform the local surgery that Conway operators and SRG need.

In [ ]:
H = example_graphs.from_tiles(example_tilesets.platonic(6), rings=2)
H.recompute_lengths_and_angles()
print(f'{len(H.vertices)} vertices, {len(H.halfedges)} half-edges, {len(H.faces)} faces')
plot_g(H, color='steelblue')
plt.show()


## Where to next

- [`01_Tilings_Euclidean`](01_Tilings_Euclidean.ipynb) — building tilings, Archimedean gallery, Conway operators.
- [`02_Curved_Geometries`](02_Curved_Geometries.ipynb) — spherical and hyperbolic tilings.
- [`03_Crease_Patterns_SRG`](03_Crease_Patterns_SRG.ipynb) — tiling → crease pattern via shrink-rotate.
- [`04_Folding_and_Overlap`](04_Folding_and_Overlap.ipynb) — flat-foldability and the folded state.
- [`05_Conway_Plus_SRG`](05_Conway_Plus_SRG.ipynb) — recipe book.
- [`06_Export_and_3D`](06_Export_and_3D.ipynb) — SVG / FOLD / STL.
